# Week 7, Days 3 and 4: Setting up an SFT run

**Lesson:** [the Colab notebook](https://colab.research.google.com/drive/1fBTm_jzrFGr88PDOFTQF7JIlQ1JW15BG) linked from [week7/day3 and 4.ipynb](../../../week7/day3%20and%204.ipynb), *Training, parts 1 and 2*. **About 30 minutes.**

**What you practice here**

- Choosing QLoRA's settings for a lite and a full run: r, alpha at twice r, and the modules to target.
- Writing the settings for `LoraConfig` and `SFTConfig` yourself: epochs, batch size and the learning-rate schedule, precision chosen by the GPU, checkpoints pushed to the Hugging Face Hub, and Weights & Biases reporting.
- Making TRL's `SFTTrainer` from the model, the data and both configurations, then training it and pushing the adapter to the Hub.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week7/day3_and_4", globals())
else:  # in Colab, or outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here, as in Colab.")

## The task

Days 3 and 4 fine-tune Llama 3.2 3B on the price data with QLoRA. The lesson sets its hyperparameters as constants,
makes a `LoraConfig` and an `SFTConfig` from them, and hands both to TRL's `SFTTrainer` with the model and the data.
Then it trains, and pushes the adapter to the Hugging Face Hub. Write that set-up as functions you can reuse, and use
them for a short trial run of your own.

peft and trl are not in the checks' Python, so the two configurations are dicts of the keyword arguments they take,
to unpack into `LoraConfig` and `SFTConfig`, and `make_trainer` takes the trainer class as a parameter. The checks
know every field the real classes take, so a misspelled setting fails here as it would there. They hand your
functions a stand-in `SFTTrainer` with the real one's parameters, and a dataset whose splits record what you
select. The stand-in rows are wrestling merchandise.

## What each call should give

```python
LoraConfig(**lora_settings(lite=True))
# the lesson's lite LoRA: r 32, alpha 64, dropout 0.1, no bias, a causal LM, on the four attention modules

args = SFTConfig(**training_settings("price-2026-10-15_09.30.00-lite",
                                     "your-hf-name/price-2026-10-15_09.30.00-lite", (7, 5)))
# a lite run on a T4: 1 epoch at batch 32 in fp16, with a checkpoint pushed to the Hub every 100 steps

trainer = make_trainer(SFTTrainer, base_model, dataset, LoraConfig(**lora_settings()), args)
# an SFTTrainer with the model, the train split, the first 500 rows of the val split, and both configurations

train_and_push(trainer, "price-2026-10-15_09.30.00-lite")
# trains, then pushes the adapter to your Hugging Face account as a private repo
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.

### 1 · `lora_settings(lite=True)`

Returns the settings of the lesson's `LoraConfig`, as a dict of keyword arguments, for a lite or a full run.

- The rank r is 32 for lite and 256 for full, alpha is twice r, and the dropout is 0.1.
- LoRA trains no biases (`"none"`), and the task is a causal language model (`"CAUSAL_LM"`).
- The modules to adapt are a list: for lite, the four attention modules, `"q_proj"`, `"k_proj"`, `"v_proj"` and
  `"o_proj"`; for full, those four and the MLP's `"gate_proj"`, `"up_proj"` and `"down_proj"`.

Use `LoraConfig`'s own names for the keys: the checks know them all, and refuse any other.

<details><summary>Hint</summary>

```python
{"r": r, "lora_alpha": r * 2, "lora_dropout": 0.1, "bias": "none", "task_type": "CAUSAL_LM",
 "target_modules": [...]}
```

</details>

In [ ]:
def lora_settings(lite=True):
    # Write lora_settings here, as described above.
    raise NotImplementedError("lora_settings is not written yet")

In [ ]:
check("lora_settings")

### 2 · `training_settings(run_name, hub_model_id, capability, lite=True, log_to_wandb=True)`

Returns the settings of the lesson's `SFTConfig`, as a dict of keyword arguments, for a lite or a full run.
`capability` is the GPU's compute capability, as `torch.cuda.get_device_capability()` gives it: `(7, 5)` on a T4,
`(8, 0)` on an A100. Use `SFTConfig`'s own names for the keys (the hint lists them). The dict holds at least:

- the output folder and the run's name, both `run_name`;
- 1 epoch at a batch size of 32 per device for lite, and 3 epochs at 256 for full, with 1 gradient accumulation
  step;
- a learning rate of 1e-4 on a `"cosine"` schedule, a warmup ratio of 0.01, the `"paged_adamw_32bit"` optimizer,
  and sequences cut at 128 tokens;
- bf16 on and fp16 off when the capability's first number is 8 or more, and the other way round below 8, as on a T4;
- saving and evaluating by `"steps"`, every 100 steps for lite and 200 for full, and logging every 5 steps for lite
  and 10 for full;
- pushing to the Hub at every save (`"every_save"`), to the private repo `hub_model_id`;
- reporting to `"wandb"`, or to the string `"none"` when `log_to_wandb` is False. The lesson writes `None` there,
  but in transformers 4.57 `None` means every installed integration, wandb included.

You may add the lesson's other settings. In the versions the lesson pins, the sequence limit is called `max_length`
and the evaluation setting `eval_strategy`; older tutorials write `max_seq_length` and `evaluation_strategy`, which
these versions refuse.

<details><summary>Hint</summary>

```python
dict(output_dir=run_name, run_name=run_name, num_train_epochs=..., per_device_train_batch_size=...,
     gradient_accumulation_steps=1, learning_rate=1e-4, lr_scheduler_type="cosine", warmup_ratio=0.01,
     optim="paged_adamw_32bit", max_length=128, bf16=..., fp16=..., save_strategy="steps", save_steps=...,
     eval_strategy="steps", eval_steps=..., logging_steps=..., push_to_hub=True, hub_model_id=hub_model_id,
     hub_strategy="every_save", hub_private_repo=True, report_to=...)
```

</details>

In [ ]:
def training_settings(run_name, hub_model_id, capability, lite=True, log_to_wandb=True):
    # Write training_settings here, as described above.
    raise NotImplementedError("training_settings is not written yet")

In [ ]:
check("training_settings")

### 3 · `make_trainer(trainer_class, model, dataset, lora_config, train_config, val_size=500)`

Makes the trainer from the model, the data and the two configurations, as the lesson does. `trainer_class` is TRL's
`SFTTrainer`, and `dataset` is what `load_dataset` returned, with `"train"` and `"val"` splits. Make one trainer with
`model`, the whole train split to train on, and the first `val_size` rows of the val split to evaluate on, selected
from the split so that they stay a dataset. Give it `lora_config` as its PEFT config and `train_config` as its
arguments. Pass everything by name: the trainer's second parameter is its arguments, not the training data. The
first `val_size` rows are enough to watch the evaluation loss.

<details><summary>Hint</summary>

```python
trainer_class(model=model, train_dataset=dataset["train"], eval_dataset=dataset["val"].select(range(val_size)),
              peft_config=lora_config, args=train_config)
```

</details>

In [ ]:
def make_trainer(trainer_class, model, dataset, lora_config, train_config, val_size=500):
    # Write make_trainer here, as described above.
    raise NotImplementedError("make_trainer is not written yet")

In [ ]:
check("make_trainer")

### 4 · `train_and_push(trainer, repo_name)`

Trains, then pushes the trained adapter to the Hub as a private repo. Train the trainer, then push its model, the
adapter it trained, to the repo `repo_name` as a private repo, as the lesson does. Return what training returned.
Push the model, not the trainer: the trainer's own push goes to the `hub_model_id` of its settings, not to
`repo_name`.

<details><summary>Hint</summary>

```python
result = trainer.train()
trainer.model.push_to_hub(repo_name, private=True)
return result
```

</details>

In [ ]:
def train_and_push(trainer, repo_name):
    # Write train_and_push here, as described above.
    raise NotImplementedError("train_and_push is not written yet")

In [ ]:
check("train_and_push")

## Check everything

The cell below runs all 9 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 7, Days 3 and 4 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. This needs a GPU, so it runs in Google Colab: push this notebook, then [open it in Colab](https://colab.research.google.com/github/starlest/llm_engineering/blob/main/plan/exercises/week7/day3_and_4.ipynb), choose a T4 runtime, and run the cells from the top. In Colab the checks are off, so run them here in your codespace. Write the steps in the 2 cells at the end.

Colab works on its own copy of the notebook. To keep the Try code you write there, use **File > Save a copy in GitHub** (to this notebook's path in your fork), then run `git pull` in your codespace before you commit there again. Your parts are already in your fork, which is what the tracker reads.

A free T4 is enough for this 30-step trial, which takes a few minutes. Besides `HF_TOKEN`, add your Weights & Biases
key to Colab's secrets as `WANDB_API_KEY` (sign up at wandb.ai). Accept the Llama 3.2 license on its Hugging Face
page first.

1. Keep the `!pip install` line in the first Try cell. Log in to Hugging Face and to Weights & Biases from Colab's
   secrets, and set `WANDB_PROJECT` to `"price"`. Choose a `RUN_NAME`, the date and time as the lesson's is, and
   start a W&B run with that name.
2. Load `ed-donner/items_prompts_lite` as `dataset`, then the tokenizer and the 4-bit base model as the lesson does,
   as `base_model`, with the compute dtype chosen by `torch.cuda.get_device_capability()`.
3. Build `LoraConfig(**lora_settings())` and `SFTConfig(**training_settings(...))`, with your Hugging Face name in
   `hub_model_id`, overriding `max_steps`, `save_steps` and `eval_steps` for a 30-step trial.
4. Make the trainer with `make_trainer`, passing TRL's `SFTTrainer`, `base_model`, `dataset`, both configurations
   and `val_size=100`. Then call `train_and_push(trainer, RUN_NAME)`, and `wandb.finish()`.
5. Find the loss curves in Weights & Biases, and the adapter, with a commit for each checkpoint, in your Hugging Face
   account. Day 5 can load it.

<details><summary>Stuck? One way to write it</summary>

```python
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1 transformers==4.57.6
import os
from datetime import datetime
import torch
import wandb
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

login(userdata.get("HF_TOKEN"), add_to_git_credential=True)
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["WANDB_PROJECT"] = "price"
wandb.login()

HF_USER = "your-hf-name"
BASE_MODEL = "meta-llama/Llama-3.2-3B"
RUN_NAME = f"price-{datetime.now():%Y-%m-%d_%H.%M.%S}-trial"
wandb.init(project="price", name=RUN_NAME)

dataset = load_dataset("ed-donner/items_prompts_lite")
capability = torch.cuda.get_device_capability()
quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True, bnb_4bit_quant_type="nf4",
                                  bnb_4bit_compute_dtype=torch.bfloat16 if capability[0] >= 8 else torch.float16)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=quant_config, device_map="auto")
base_model.generation_config.pad_token_id = tokenizer.pad_token_id
```

```python
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

lora = LoraConfig(**lora_settings(lite=True))
trial = {"max_steps": 30, "save_steps": 15, "eval_steps": 15}  # a short trial: the lesson's lite run is a full epoch
args = SFTConfig(**{**training_settings(RUN_NAME, f"{HF_USER}/{RUN_NAME}", capability), **trial})
trainer = make_trainer(SFTTrainer, base_model, dataset, lora, args, val_size=100)
print(train_and_push(trainer, RUN_NAME))
wandb.finish()
```

</details>

In [ ]:
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1 transformers==4.57.6
# Steps 1 and 2: log in to Hugging Face and W&B, then load the data, the tokenizer and the 4-bit base model

In [ ]:
# Steps 3 to 5: build the configurations, make the trainer, then train for 30 steps and push